# التدخل البشري في الحلقة: بوابات ما قبل الإجراء، تصنيف المخاطر، وتسجيل التدقيق

توضح وثيقة README لهذه الدرس التدخل البشري في الحلقة بمقطع قصير يطلب من المستخدم إما `الموافقة` أو `الرفض` بعد أن ينتج الوكيل استجابة. هذا النمط هو نقطة بداية جيدة، لكن تطبيقات التدخل البشري في الإنتاج عادةً ما تحتاج إلى ثلاث عناصر إضافية:

1. **بوابة ما قبل الإجراء** تعمل **قبل** تنفيذ الوكيل لخطوة محفوفة بالمخاطر، حتى تبقى التكلفة، وعدم القابلية للعكس، والكمون تحت السيطرة.
2. **تصنيف المخاطر**، بحيث يتم تنفيذ الإجراءات منخفضة الخطورة تلقائيًا، ويتم الموافقة على الإجراءات متوسطة الخطورة دفعة واحدة، وتُعلق فقط الإجراءات عالية الخطورة على تدخل بشري.
3. **سجل تدقيق مع حلقة مراجعة**، بحيث يتم تسجيل كل قرار بوابة كـ JSONL، ويعيد الرفض تحفيز الوكيل مع سبب منظم بدلًا من مجرد طباعة `جارٍ المراجعة...`.

يبني هذا الدفتر كلًا من هذه العناصر فوق نفس البدائيات كما في `06-system-message-framework.ipynb`. ويعمل من البداية للنهاية في وضع `DEMO_MODE = True` (دون الحاجة إلى إدخال تفاعلي) أو مع مطالبات `input()` الحقيقية عند `DEMO_MODE = False`. ملاحظة: في وضع DEMO_MODE يتم برمجة إعادة المحاولة للهدف الثالث حتى تصبح آلية الحلقة مرئية من البداية للنهاية. أما إعادة التصنيف المدفوعة بالمراجعة الحقيقية فتتطلب `DEMO_MODE = False` ومشغلًا.

**خارجة عن النطاق (يُتعامل معها في دروس أخرى):** المصادقة والتحكم في الوصول (درس 06 README تهديد 2)، وسيط استدعاء الأدوات (درس 14 MAF تحليل عميق)، وأنماط الجدال متعددة الوكلاء.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## النمط 1: بوابة ما قبل الفعل

مقتطف HITL في ملف README يستدعي الوكيل أولاً، ثم يطلب من المستخدم الموافقة على النتيجة. هذه هي تدفق **ما بعد الفعل**. لقد تم تنفيذ الوكيل بالفعل، لذا تم دفع تكلفة استدعاء LLM بالفعل، وأي تأثير جانبي (رسالة بريد إلكتروني مرسلة، صف قاعدة بيانات مكتوب، تعليق منشور) قد حدث بالفعل.

تدفق **ما قبل الفعل** يُدخل البوابة قبل أن ينفذ الوكيل الخطوة الخطرة. يقترح الوكيل الإجراء، تقرر البوابة ما إذا كان سيتم التنفيذ، وفقط عند الموافقة يحدث التأثير الجانبي.

| الجانب | الموافقة بعد الفعل (مقتطف README) | بوابة ما قبل الفعل (هذا الدفتر) |
|---|---|---|
| متى يتم تنفيذ الموافقة؟ | بعد أن ينتج الوكيل المخرجات | قبل تنفيذ أي تأثير جانبي |
| تكلفة LLM عند الرفض | تم دفعها بالفعل | تُدفع فقط على الاقتراح، وليس على الفعل |
| التأثيرات الجانبية غير القابلة للعكس | ممكنة (لقد حدث الإجراء بالفعل) | ممنوعة |
| وضوح التدقيق | الموافقة هي عبارة عن بيان طباعة | الموافقة هي سجل JSON مع طابع زمني، الإجراء، السبب |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## النموذج 2: تصنيف المخاطر

ليس كل إجراء يحتاج إلى موافقة بشرية. البحث فقط للقراءة مقابل واجهة برمجة تطبيقات عامة له مخاطر مختلفة عن إرسال بريد إلكتروني للعميل. التعامل مع الاثنين بنفس الطريقة يضيع انتباه المشغل ويبطئ الوكيل.

نموذج بسيط من 3 مستويات:

| المستوى | أمثلة | تسلسل الموافقة |
|---|---|---|
| `منخفض` (قراءة فقط) | البحث في قاعدة معارف، الاطلاع على خيارات الرحلات، جلب صفحة ويب عامة | تنفيذ تلقائي، مسجل للمراجعة |
| `متوسط` (تغيير قليل التكلفة) | تخزين نتيجة مؤقتة، زيادة عداد، جدولة تذكير | تنفيذ تلقائي، لكن مع مراجعة يومية مجمعة |
| `مرتفع` (موجه للجمهور أو لا رجعة فيه) | إرسال بريد إلكتروني، شحن بطاقة، النشر في قناة عامة | التوقف لحين الموافقة البشرية |

هذا هو تصنيف واحد. غالبًا ما تستخدم أنظمة الإنتاج مستويات أكثر تفصيلاً (مثلاً، مستويات أذونات AWS IAM، مستويات الوصول القائمة على الأدوار). النسخة ذات الثلاث مستويات أدناه هي أصغر نسخة مفيدة لوكيل يدمج الأفعال للقراءة فقط والأفعال الجانبية.

يصنف المُصنف أدناه باستخدام قواعد تعتمد على الكلمات المفتاحية لكي يبقى العرض التجريبي حتمي ورخيص. في نظام انتاجي ستستبدل هذا بمصنف متعلم أو محرك سياسة.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## النمط 3: سجل التدقيق وحلقة المراجعة

لا يعد `print("Response approved.")` سجل تدقيق. من أجل الثقة، يجب تسجيل كل قرار بوابة كحدث منظم يمكنك في وقت لاحق الاستعلام عنه، أو إعادة تشغيله، أو إرفاقه بمراجعة حادث.

جزأين:

1. **JSONL للكتابة فقط.** سطر واحد لكل قرار، مع الطابع الزمني، والإجراء، والمستوى، والقرار، والسبب. سهل البحث (grep) وسهل الإرسال إلى مخزن سجل حقيقي لاحقًا.
2. **حلقة المراجعة عند الرفض.** عندما تعيد البوابة `deny`، يعيد العميل طلبه بنفسه مع سبب الرفض في السياق، بحيث يمكن للاقتراح التالي تجنب المشكلة.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## موارد إضافية

تنفذ عدة مشاريع عامة أخرى تنويعات من أنماط الإنسان في الحلقة هذه. قارن الأساليب للعثور على ما يناسب مجموعتك:

- **LangChain** أغلفة أدوات الإنسان في الحلقة ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): أغلفة أدوات يتم إدخالها توقف التنفيذ لمدخلات بشرية.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); أُعيد هيكلة AutoGen v0.4+ هذا): يستخدم دور وكيل لتمثيل الإنسان بشكل خاص في المحادثات متعددة الوكلاء.
- **إطار عمل وكيل مايكروسوفت (MAF)** وسيط استدعاء الوظائف ([docs](https://learn.microsoft.com/agent-framework/)): وسيط يعمل حول كل استدعاء أداة/وظيفة، مناسب لمنطق البوابة وتدفقات الموافقة.

كل مشروع يتعامل مع الأنماط الفرعية الثلاثة بشكل مختلف: LangChain يغلفها كأدوات، AutoGen يستخدم دور وكيل، وإطار عمل وكيل مايكروسوفت يستخدم وسيط استدعاء الوظائف. اقرأ واحدًا أو اثنين من التطبقيات من البداية للنهاية قبل اختيار تصميم لوكيلك الخاص.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
